In [ ]:
# %pip install anthropic python-dotenv

%pip install -q -U google-genai python-dotenv

In [ ]:
# Load env variables from .env file
from dotenv import load_dotenv
load_dotenv()



In [ ]:
#Create an API client
# from anthropic import Anthropic

# client = Anthropic()
# model = "claude-haiku-4-5-20251001"
from google import genai
import os

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)
chat = client.chats.create(
    model="gemini-3.6-flash",
)
response = chat.send_message(
    "Hello, how are you?"
)

print(response.text)

In [ ]:
from google.genai import types

def add_user_msg(msg, text):
    user_message = {"role": "user", "content": text}
    msg.append(user_message)

def add_ai_msg(msg, text):
    ai_msg = {"role": "assistant", "content": text}
    msg.append(ai_msg)

#Make a request : CLaude
def claudeChat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages
    )
    return message.content[0].text

def geminiChat(contents: str):
    response = chat.send_message(
        contents.__str__(),
        config=types.GenerateContentConfig(max_output_tokens = 1000)
    )
    return response.text

#Streaming AI response
def geminiStreamChat(contents: str, system = None):
    config=types.GenerateContentConfig(max_output_tokens = 1000)

    if system:
            config = types.GenerateContentConfig(
                    system_instruction=system,
                    max_output_tokens = 1000
                )
    for chunk in chat.send_message_stream(
        contents.__str__(),
        config
    ):
        if chunk.text:
            print(chunk.text, end="", flush=True)

    print("\n")

#Symtem Prompts
def SystemChat(contents, system = None):
    params = {
        "contents" : contents.__str__(),
    }
    if system:
        params["config"] = types.GenerateContentConfig(
            system_instruction=system,
            max_output_tokens = 1000
        )
    
    response = chat.send_message_stream(**params)
    return response.text


In [ ]:
#Make an initial list of messages
messages = []

while True:
    user_input = input("> ")
    print(">", user_input)
    add_user_msg(messages, user_input)
    answer = geminiStreamChat(messages)
    add_ai_msg(messages, answer)
    print('************')
    print(answer)
    print('************')



In [ ]:
messages = []

system = "You are a tech lead in a IT service provider company. You are taking interview of Full stack developer with 2+ experience."

add_user_msg(messages, "What are the node js top 5 interview questions, with one liner answer?")
answer = geminiStreamChat(messages, system)

answer